# Tennis Vision — Colab Runner
Analyzes a tennis video: ball tracking, player detection, pose, speed & in/out calls.

---
## Sections
- **[A] Setup** — clone, install, download models (run once)
- **[B] Process Video** — analyze your YouTube Shorts video
- **[C] Fine-tune Ball Detector** — improve accuracy with real tennis data
- **[D] Fine-tune Court Detector** — improve in/out & speed accuracy
---
> **First time?** Run Section A top to bottom, then B.
> **Already set up?** Just run Cell A1 (`git pull`) then go to B.

## Section A — Setup

In [ ]:
# ── A1: Clone repo (skips if already cloned, pulls latest if exists) ────────
import os

if not os.path.exists('/content/tennis_vision'):
    !git clone https://github.com/bozanctn/tennis_vision.git /content/tennis_vision
    print('Cloned.')
else:
    print('Repo already exists — pulling latest changes...')
    !git -C /content/tennis_vision pull origin main

%cd /content/tennis_vision

In [ ]:
# ── A2: Install dependencies ─────────────────────────────────────────────────
!pip install -q -r requirements.txt
!pip install -q roboflow   # for automatic dataset download
print('\nAll dependencies installed.')

In [ ]:
# ── A3: Download base YOLOv8 model ───────────────────────────────────────────
!python scripts/download_models.py

## Section B — Process Your Video

In [ ]:
# ── B1: Process YouTube Shorts video ─────────────────────────────────────────
YOUTUBE_URL = 'https://www.youtube.com/shorts/08-cKvAJMVo'
OUTPUT_FILE = '/content/tennis_vision/result.mp4'

!python scripts/process_video.py \
    --input "{YOUTUBE_URL}" \
    --output "{OUTPUT_FILE}" \
    --device cuda

print(f'\nDone! Output saved to: {OUTPUT_FILE}')

In [ ]:
# ── B2: Preview result inside Colab ──────────────────────────────────────────
from IPython.display import HTML
from base64 import b64encode
import os

output_file = '/content/tennis_vision/result.mp4'

if os.path.exists(output_file):
    playable = output_file.replace('.mp4', '_preview.mp4')
    !ffmpeg -y -i "{output_file}" -vcodec libx264 -acodec aac "{playable}" -loglevel quiet
    with open(playable, 'rb') as f:
        video_data = f.read()
    b64 = b64encode(video_data).decode()
    display(HTML(f'''
        <video width="720" controls>
            <source src="data:video/mp4;base64,{b64}" type="video/mp4">
        </video>
    '''))
else:
    print('Output file not found — check B1 for errors.')

In [ ]:
# ── B3: Download result to your computer ─────────────────────────────────────
from google.colab import files
files.download('/content/tennis_vision/result.mp4')

In [ ]:
# ── B4 (optional): Save to Google Drive ──────────────────────────────────────
from google.colab import drive
import shutil
drive.mount('/content/drive')
shutil.copy('/content/tennis_vision/result.mp4', '/content/drive/MyDrive/tennis_result.mp4')
print('Saved to Google Drive: tennis_result.mp4')

## Section C — Fine-tune Ball Detector

This improves ball detection accuracy significantly.

**You need a free Roboflow API key:**
1. Go to [roboflow.com](https://roboflow.com) → Sign up (free)
2. Go to **Settings → Roboflow API** → copy your key
3. Paste it in cell C1 below

That's it — dataset download, training, and config update are fully automatic.

In [ ]:
# ── C1: Download tennis ball dataset (automatic) ─────────────────────────────
# Paste your free Roboflow API key below (roboflow.com → Settings → API)
ROBOFLOW_API_KEY = 'PASTE_YOUR_KEY_HERE'

from roboflow import Roboflow
import os

os.makedirs('/content/tennis_vision/data', exist_ok=True)
os.chdir('/content/tennis_vision/data')

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# This is a public tennis ball dataset on Roboflow Universe (~1700 images)
# It includes ball images at various distances, lighting, and motion blur levels
project = rf.workspace('tennis-nimqk').project('tennis-ball-detection-dkqrp')
dataset = project.version(1).download('yolov8')

BALL_DATASET_PATH = dataset.location
print(f'\nDataset downloaded to: {BALL_DATASET_PATH}')
print(f'Dataset YAML: {BALL_DATASET_PATH}/data.yaml')

os.chdir('/content/tennis_vision')

In [ ]:
# ── C2: Inspect dataset stats ────────────────────────────────────────────────
import yaml, os
from pathlib import Path

with open(f'{BALL_DATASET_PATH}/data.yaml') as f:
    data_cfg = yaml.safe_load(f)

print('Classes  :', data_cfg.get('names'))
print('Num classes:', data_cfg.get('nc'))

train_imgs = list(Path(f'{BALL_DATASET_PATH}/train/images').glob('*'))
val_imgs   = list(Path(f'{BALL_DATASET_PATH}/valid/images').glob('*'))
print(f'Train images: {len(train_imgs)}')
print(f'Val   images: {len(val_imgs)}')

In [ ]:
# ── C3: Train ball detector (~15-20 min on T4 GPU) ───────────────────────────
# Starting from YOLOv8n pretrained weights (transfer learning)
# epochs=50 is enough for a small dataset; increase to 100 for better accuracy

!yolo train \
  model=yolov8n.pt \
  data="{BALL_DATASET_PATH}/data.yaml" \
  epochs=50 \
  imgsz=640 \
  batch=16 \
  project=/content/tennis_vision/models \
  name=ball_detector \
  device=0 \
  exist_ok=True

In [ ]:
# ── C4: Check training results ───────────────────────────────────────────────
from ultralytics import YOLO
from IPython.display import Image, display
import os

best_weights = '/content/tennis_vision/models/ball_detector/weights/best.pt'

if os.path.exists(best_weights):
    model = YOLO(best_weights)
    metrics = model.val(data=f'{BALL_DATASET_PATH}/data.yaml', device=0)
    print(f'\nmAP@50     : {metrics.box.map50:.3f}  (aim for >0.70)')
    print(f'mAP@50-95  : {metrics.box.map:.3f}')
    print(f'Precision  : {metrics.box.mp:.3f}')
    print(f'Recall     : {metrics.box.mr:.3f}')

    # Show training curves
    results_img = '/content/tennis_vision/models/ball_detector/results.png'
    if os.path.exists(results_img):
        display(Image(results_img, width=900))
else:
    print('Weights not found — check C3 for errors.')

In [ ]:
# ── C5: Update config to use the fine-tuned ball model ───────────────────────
import yaml

config_path = '/content/tennis_vision/config/config.yaml'
with open(config_path) as f:
    cfg = yaml.safe_load(f)

cfg['models']['ball_model_path'] = 'models/ball_detector/weights/best.pt'

with open(config_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('Config updated!')
print('ball_model_path ->', cfg['models']['ball_model_path'])
print('\nNow re-run Section B to process your video with the improved model.')

## Section D — Fine-tune Court Detector

Improves court keypoint accuracy → better in/out calls & more accurate speed.

Uses YOLOv8-**pose** mode: it predicts 14 keypoints (court line intersections) instead of bounding boxes.

Same requirement: free Roboflow API key from Section C.

In [ ]:
# ── D1: Download tennis court keypoint dataset (automatic) ───────────────────
# Make sure ROBOFLOW_API_KEY is set (run C1 first if not)

from roboflow import Roboflow
import os

os.makedirs('/content/tennis_vision/data', exist_ok=True)
os.chdir('/content/tennis_vision/data')

rf = Roboflow(api_key=ROBOFLOW_API_KEY)

# Public tennis court line/keypoint dataset
project = rf.workspace('tennis-court-zqmfm').project('tennis-court-detection-uq5vc')
court_dataset = project.version(1).download('yolov8')

COURT_DATASET_PATH = court_dataset.location
print(f'\nCourt dataset downloaded to: {COURT_DATASET_PATH}')

os.chdir('/content/tennis_vision')

In [ ]:
# ── D2: Train court detector (~20-30 min on T4 GPU) ──────────────────────────
# Uses higher resolution (1280) because court lines are thin and need detail

!yolo train \
  model=yolov8n.pt \
  data="{COURT_DATASET_PATH}/data.yaml" \
  epochs=80 \
  imgsz=1280 \
  batch=8 \
  project=/content/tennis_vision/models \
  name=court_detector \
  device=0 \
  exist_ok=True

In [ ]:
# ── D3: Update config to use fine-tuned court model ──────────────────────────
import yaml

config_path = '/content/tennis_vision/config/config.yaml'
with open(config_path) as f:
    cfg = yaml.safe_load(f)

cfg['models']['court_model_path'] = 'models/court_detector/weights/best.pt'

with open(config_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('Config updated!')
print('court_model_path ->', cfg['models']['court_model_path'])
print('\nNow re-run Section B to process your video with both improved models.')

## Section E — Save Fine-tuned Models to Google Drive

Colab resets every session — save your trained models to Drive so you don't have to retrain every time.

In [ ]:
# ── E1: Save models to Google Drive ──────────────────────────────────────────
from google.colab import drive
import shutil, os

drive.mount('/content/drive')

drive_models_dir = '/content/drive/MyDrive/tennis_vision_models'
os.makedirs(drive_models_dir, exist_ok=True)

# Save ball detector
ball_weights = '/content/tennis_vision/models/ball_detector/weights/best.pt'
if os.path.exists(ball_weights):
    shutil.copy(ball_weights, f'{drive_models_dir}/ball_detector.pt')
    print('Saved ball_detector.pt to Drive')

# Save court detector
court_weights = '/content/tennis_vision/models/court_detector/weights/best.pt'
if os.path.exists(court_weights):
    shutil.copy(court_weights, f'{drive_models_dir}/court_detector.pt')
    print('Saved court_detector.pt to Drive')

print(f'\nAll models saved to: {drive_models_dir}')

In [ ]:
# ── E2: Load models from Drive (use this in future sessions) ─────────────────
from google.colab import drive
import shutil, os, yaml

drive.mount('/content/drive')

drive_models_dir = '/content/drive/MyDrive/tennis_vision_models'
local_models_dir = '/content/tennis_vision/models'
os.makedirs(local_models_dir, exist_ok=True)

# Restore ball detector
ball_src = f'{drive_models_dir}/ball_detector.pt'
if os.path.exists(ball_src):
    shutil.copy(ball_src, f'{local_models_dir}/ball_detector.pt')
    print('Loaded ball_detector.pt from Drive')

# Restore court detector
court_src = f'{drive_models_dir}/court_detector.pt'
if os.path.exists(court_src):
    shutil.copy(court_src, f'{local_models_dir}/court_detector.pt')
    print('Loaded court_detector.pt from Drive')

# Update config to point to restored models
config_path = '/content/tennis_vision/config/config.yaml'
with open(config_path) as f:
    cfg = yaml.safe_load(f)

if os.path.exists(f'{local_models_dir}/ball_detector.pt'):
    cfg['models']['ball_model_path'] = 'models/ball_detector.pt'
if os.path.exists(f'{local_models_dir}/court_detector.pt'):
    cfg['models']['court_model_path'] = 'models/court_detector.pt'

with open(config_path, 'w') as f:
    yaml.dump(cfg, f, default_flow_style=False)

print('\nConfig updated — ready to run Section B!')